# Chapter 9 — Exercises: Worked Solutions

Worked solutions for Chapter 9 (Water–Hydrocarbon Systems).

**Exercises:**
1. Water content of natural gas vs pressure
2. Hydrocarbon solubility in water
3. Three-phase VLLE flash

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    SystemSrkCPAstatoil = ns.SystemSrkCPAstatoil
    SystemSrkEos = ns.SystemSrkEos
    ThermodynamicOperations = ns.ThermodynamicOperations

---
## Exercise 9.1 — Water Content of Natural Gas

**Problem:** Calculate the water content (mg/Sm\u00b3) of a lean natural
gas (90% CH\u2084, 5% C\u2082H\u2086, 3% C\u2083H\u2088, 2% CO\u2082) at 40\u00b0C
for pressures from 20 to 150 bar using CPA.

### Solution

In [3]:
pressures = np.arange(20, 155, 5)
water_content = []  # mg/Sm3

for P in pressures:
    try:
        f = SystemSrkCPAstatoil(273.15 + 40.0, float(P))
        f.addComponent("methane", 0.90)
        f.addComponent("ethane", 0.05)
        f.addComponent("propane", 0.03)
        f.addComponent("CO2", 0.02)
        f.addComponent("water", 0.001)  # trace
        f.setMixingRule(10)
        f.setMultiPhaseCheck(True)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        y_w = float(f.getPhase("gas").getComponent("water").getx())
        # Convert mole fraction to mg/Sm3
        # At standard conditions: 1 Sm3 = 44.6 mol (ideal)
        water_mg = y_w * 44.6 * 18.015 * 1000  # mg/Sm3
        water_content.append(water_mg)
    except Exception:
        water_content.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.semilogy(pressures[:len(water_content)], water_content, color=BLUE, linewidth=1.5)
ax.axhline(y=50, color="red", linestyle="--", alpha=0.5, linewidth=0.8)
ax.annotate("Pipeline spec (~50 mg/Sm\u00b3)", xy=(100, 55), fontsize=7, color="red")
ax.set_xlabel("Pressure (bar)"); ax.set_ylabel("Water content (mg/Sm\u00b3)")
ax.set_title("Exercise 9.1: Natural gas water content")
ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
save(fig, "fig_ch09_ex01_water_content.png")

Saved: fig_ch09_ex01_water_content.png


**Answer:** Water content decreases rapidly with increasing pressure.
At typical pipeline conditions (70–100 bar), the saturated water content
is well above the 50 mg/Sm\u00b3 specification, demonstrating why
dehydration (e.g., TEG absorption) is essential.

---
## Exercise 9.2 — Methane Solubility in Water

**Problem:** Calculate the solubility of methane in water at 25\u00b0C
from 10 to 200 bar. Compare with Henry's law.

### Solution

In [4]:
pressures_ch4 = np.arange(10, 205, 10)
x_ch4_cpa = []

for P in pressures_ch4:
    try:
        f = SystemSrkCPAstatoil(273.15 + 25.0, float(P))
        f.addComponent("methane", 0.99)
        f.addComponent("water", 0.01)
        f.setMixingRule(10)
        f.setMultiPhaseCheck(True)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        nph = int(f.getNumberOfPhases())
        for ph in range(nph):
            pt = str(f.getPhase(ph).getPhaseTypeName()).lower()
            if "aqueous" in pt or ("liquid" in pt and ph > 0):
                x_ch4_cpa.append(float(f.getPhase(ph).getComponent("methane").getx()) * 1e4)
                break
        else:
            x_ch4_cpa.append(np.nan)
    except Exception:
        x_ch4_cpa.append(np.nan)

# Henry's law: x = P / H, H_CH4 = 4.19e4 bar at 25 C
H_ch4 = 4.19e4  # bar
x_henry = pressures_ch4 / H_ch4 * 1e4  # x10^4

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures_ch4[:len(x_ch4_cpa)], x_ch4_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.plot(pressures_ch4, x_henry, color=ORANGE, linewidth=1.2, linestyle="--", label="Henry's law")
ax.set_xlabel("Pressure (bar)"); ax.set_ylabel(r"$x_{CH_4} \times 10^4$")
ax.set_title("Exercise 9.2: CH\u2084 solubility in water")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch09_ex02_ch4_solubility.png")

Saved: fig_ch09_ex02_ch4_solubility.png


**Answer:** Henry's law (linear) is accurate at low pressures but
over-predicts solubility at higher pressures because it ignores gas-phase
non-ideality and the pressure dependence of Henry's constant. CPA
captures this curvature naturally through the equation of state.

---
## Exercise 9.3 — Three-Phase Flash

**Problem:** Demonstrate three-phase VLLE for a methane–n-octane–water
system at 25\u00b0C and 50 bar.

### Solution

In [5]:
f = SystemSrkCPAstatoil(273.15 + 25.0, 50.0)
f.addComponent("methane", 0.80)
f.addComponent("n-octane", 0.10)
f.addComponent("water", 0.10)
f.setMixingRule(10)
f.setMultiPhaseCheck(True)
ops = ThermodynamicOperations(f)
ops.TPflash()
f.initProperties()

nph = int(f.getNumberOfPhases())
print(f"Number of phases: {nph}")
print()

for ph in range(nph):
    phase = f.getPhase(ph)
    pt = str(phase.getPhaseTypeName())
    beta = float(phase.getBeta())
    print(f"Phase {ph}: {pt} (fraction = {beta:.4f})")
    for comp in ["methane", "n-octane", "water"]:
        xi = float(phase.getComponent(comp).getx())
        print(f"  x_{comp} = {xi:.6f}")
    print()

Number of phases: 3

Phase 0: gas (fraction = 0.7756)
  x_methane = 0.997820
  x_n-octane = 0.001417
  x_water = 0.000763

Phase 1: oil (fraction = 0.1249)
  x_methane = 0.208017
  x_n-octane = 0.791549
  x_water = 0.000433

Phase 2: aqueous (fraction = 0.0995)
  x_methane = 0.001132
  x_n-octane = 0.000000
  x_water = 0.998868



**Answer:** The system splits into three phases: a gas (mostly methane),
a hydrocarbon liquid (mostly octane), and an aqueous phase (mostly water).
CPA correctly predicts the very low mutual solubility between water and
hydrocarbons while accurately describing the gas–liquid equilibrium.
This three-phase behavior is critical for oil & gas production.